# Soro-Kev-Omni: multimodal slice (spoken and screenshot requests)

Runs unattended on Kaggle T4 x2 after `soro-omni-data-pass2` finishes (attach it as an input).
The tool catalog stays text and is cached once; requests arrive as speech (Kokoro TTS of held-out
requests) or with a screenshot (OS-Atlas UI rows). Backbone: Gemma 4 E2B, 4-bit, fp32 compute on T4.

1. `media_check.py`: exactness with speech requests on the real backbone (order, composed caches, two-level)
2. `parent_check.py` for Gemma 4 E2B (block exactness with the per-layer-input fix)
3. P0 on Gemma 4 E2B with media rows: KM (Matryoshka gist pointer) on GPU 0, A (Kev-style pointer) on GPU 1
4. Results zip: `speech` vs `speech_text` accuracy is the text-to-speech gap; `ui` is screenshot pointing

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/nisaral/Soro-Boro.git"
BRANCH = "upp-p0"
WORK = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
REPO = f"{WORK}/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONPATH"] = f"{REPO}/veyra/python/src:" + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, f"{REPO}/veyra/python/src")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
import sys, subprocess, importlib
for pkg in ['transformers>=5.0', 'peft', 'bitsandbytes>=0.46.1', 'accelerate', 'huggingface_hub', 'soundfile',
            'datasets', 'sentence-transformers']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], capture_output=True, text=True)
    print(("ok   " if r.returncode == 0 else "FAIL ") + pkg + ("" if r.returncode == 0 else " :: " + r.stderr.strip()[-300:]))
importlib.invalidate_caches()
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "GPUs", torch.cuda.device_count())

In [ ]:
# Locate the data pass 2 outputs (rows + wav + screenshots) wherever Kaggle mounted them.
import glob, json, os
builds = sorted(glob.glob("/kaggle/input/**/omni_data/build/train.jsonl", recursive=True))
assert builds, "attach the soro-omni-data-pass2 output (Add input -> Notebook output)"
BUILD = os.path.dirname(builds[0])
DATA_ROOT = BUILD.split("/omni_data/")[0]
ROWS = ",".join(sorted(glob.glob(f"{BUILD}/*.jsonl")))
print("build:", BUILD); print("rows:", ROWS); print("media root:", DATA_ROOT)
n_audio = n_img = 0
for f in ROWS.split(","):
    for line in open(f):
        r = json.loads(line); n_audio += bool(r.get("audio")); n_img += bool(r.get("image"))
print("audio rows", n_audio, "image rows", n_img)
print("wav files", len(glob.glob(f"{DATA_ROOT}/**/*.wav", recursive=True)))

In [ ]:
# 1) exactness with a real speech clip on the real backbone
import glob
wavs = sorted(glob.glob(f"{DATA_ROOT}/**/*.wav", recursive=True))
pngs = sorted(glob.glob(f"{DATA_ROOT}/**/*.png", recursive=True))
args = ["--wav", wavs[0]] if wavs else []
args += ["--image", pngs[0]] if pngs else []
!CUDA_VISIBLE_DEVICES=0 python veyra/training/media_check.py --model google/gemma-4-E2B --four-bit {' '.join(args)} --out runs/media_check.json

In [ ]:
# 2) block exactness / latency for Gemma 4 E2B (reruns the pass-1 check with the per-layer-input fix)
!CUDA_VISIBLE_DEVICES=0 python veyra/training/parent_check.py --models google/gemma-4-E2B --four-bit google/gemma-4-E2B --out runs/parent_check_gemma.json

In [ ]:
%%bash -s "$ROWS" "$DATA_ROOT"
# 3) Gemma 4 E2B with spoken + screenshot rows: KM on GPU 0, A on GPU 1 (same data, seed, steps)
ROWS="$1"; ROOT="$2"
for spec in "0 KM" "1 A"; do set -- $spec
  CUDA_VISIBLE_DEVICES=$1 python veyra/training/p0_upp_ablation.py --base-model google/gemma-4-E2B --load-in-4bit \
    --configs $2 --media --media-rows "$ROWS" --media-root "$ROOT" --media-train 2000 \
    --train-rows 3000 --eval-rows 400 --batch 4 --max-seq-len 1100 --log-every 50 \
    --out runs/omni_slice_$2 > slice_$2.log 2>&1 &
done
wait
tail -n 6 slice_KM.log slice_A.log

In [ ]:
import json, glob
for f in sorted(glob.glob("runs/omni_slice_*/results.json")):
    r = json.load(open(f))
    for k, v in r.items():
        ev = v.get("eval", {})
        print(k, {s: round(ev[s]["acc"], 3) for s in ev if "acc" in ev[s]})
!zip -qr /kaggle/working/omni_slice_results.zip runs/media_check.json runs/parent_check_gemma.json runs/omni_slice_* slice_*.log
print("omni_slice_results.zip written")